In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

# --- Paths ---
PROJECT_ROOT = "/content/drive/MyDrive/credit-risk"
DATA_PROCESSED = f"{PROJECT_ROOT}/data/processed"
MODELS_PATH    = f"{PROJECT_ROOT}/models"
FIGURES_PATH   = f"{PROJECT_ROOT}/reports/figures"
SRC_PATH       = f"{PROJECT_ROOT}/src"

sys.path.insert(0, SRC_PATH)

print("✅ Imports and paths ready")

✅ Imports and paths ready


In [3]:
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# --- Load selected features (for tree models) ---
df = pd.read_csv(f"{DATA_PROCESSED}/df_selected.csv")

X = df.drop(columns=['SeriousDlqin2yrs'])
y = df['SeriousDlqin2yrs']

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# --- SMOTE on training set only ---
smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("X_train_sm shape:", X_train_sm.shape)
print("X_test shape:    ", X_test.shape)
print(f"Defaults in train after SMOTE: {y_train_sm.sum()} ({y_train_sm.mean()*100:.2f}%)")

X_train_sm shape: (223958, 11)
X_test shape:     (30000, 11)
Defaults in train after SMOTE: 111979 (50.00%)


In [4]:
# --- Load log-transformed features (for logistic regression meta-learner) ---
df_log = pd.read_csv(f"{DATA_PROCESSED}/df_log_features.csv")

X_log = df_log.drop(columns=['SeriousDlqin2yrs'])
y_log = df_log['SeriousDlqin2yrs']

# --- Same split using same random_state to guarantee alignment ---
X_train_log, X_test_log, y_train_log, y_test_log = train_test_split(
    X_log, y_log, test_size=0.2, random_state=42, stratify=y_log
)

# --- SMOTE on log-transformed training set ---
X_train_log_sm, y_train_log_sm = smote.fit_resample(X_train_log, y_train_log)

print("X_train_log_sm shape:", X_train_log_sm.shape)
print("X_test_log shape:    ", X_test_log.shape)
print(f"Columns: {X_log.columns.tolist()}")

X_train_log_sm shape: (223958, 3)
X_test_log shape:     (30000, 3)
Columns: ['log_MonthlyIncome', 'log_DebtRatio', 'log_RevolvingUtilizationOfUnsecuredLines']


In [5]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report
)

def evaluate_model(name, model, X_test, y_test):
    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc  = average_precision_score(y_test, y_proba)
    f1      = f1_score(y_test, y_pred)

    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"{'='*40}")
    print(f"  ROC-AUC : {roc_auc:.4f}")
    print(f"  PR-AUC  : {pr_auc:.4f}  ← primary metric")
    print(f"  F1      : {f1:.4f}")
    print(f"\n{classification_report(y_test, y_pred)}")

    return {"Model": name, "ROC-AUC": roc_auc, "PR-AUC": pr_auc, "F1": f1}

In [7]:
!pip install optuna catboost -q
print("✅ Optuna and CatBoost ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
✅ Optuna and CatBoost ready


In [8]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_xgb(trial):
    params = {
        "n_estimators"      : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"         : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"     : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"         : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree"  : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight"  : trial.suggest_int("min_child_weight", 1, 10),
        "random_state"      : 42,
        "verbosity"         : 0,
        "eval_metric"       : "aucpr",
    }
    model = XGBClassifier(**params)
    model.fit(X_train_sm, y_train_sm)
    y_proba = model.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, y_proba)

study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f"\n✅ Best XGBoost PR-AUC: {study_xgb.best_value:.4f}")
print(f"   Best params: {study_xgb.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Best XGBoost PR-AUC: 0.3451
   Best params: {'n_estimators': 291, 'max_depth': 10, 'learning_rate': 0.012168849729434708, 'subsample': 0.6199915172596526, 'colsample_bytree': 0.9652638607771129, 'min_child_weight': 2}


In [9]:
from lightgbm import LGBMClassifier

def objective_lgbm(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"       : trial.suggest_int("max_depth", 3, 10),
        "learning_rate"   : trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "num_leaves"      : trial.suggest_int("num_leaves", 20, 150),
        "random_state"    : 42,
        "verbosity"       : -1,
    }
    model = LGBMClassifier(**params)
    model.fit(X_train_sm, y_train_sm)
    y_proba = model.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, y_proba)

study_lgbm = optuna.create_study(direction="maximize")
study_lgbm.optimize(objective_lgbm, n_trials=50, show_progress_bar=True)

print(f"\n✅ Best LightGBM PR-AUC: {study_lgbm.best_value:.4f}")
print(f"   Best params: {study_lgbm.best_params}")

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Best LightGBM PR-AUC: 0.3571
   Best params: {'n_estimators': 367, 'max_depth': 4, 'learning_rate': 0.25614513269902417, 'subsample': 0.9678428280283313, 'colsample_bytree': 0.5746492798628466, 'min_child_samples': 56, 'num_leaves': 107}


In [10]:
from catboost import CatBoostClassifier

def objective_cat(trial):

    params = {
        "iterations": trial.suggest_int(
            "iterations", 100, 500
        ),
        "depth": trial.suggest_int(
            "depth", 3, 10
        ),
        "learning_rate": trial.suggest_float(
            "learning_rate", 0.01, 0.3, log=True
        ),
        "l2_leaf_reg": trial.suggest_float(
            "l2_leaf_reg", 1, 10
        ),
        "random_strength": trial.suggest_float(
            "random_strength", 0, 5
        ),
        "loss_function": "Logloss",
        "verbose": 0,
        "random_state": 42,
    }

    model = CatBoostClassifier(**params)

    model.fit(X_train_sm, y_train_sm)

    y_proba = model.predict_proba(X_test)[:, 1]

    return average_precision_score(
        y_test,
        y_proba
    )


study_cat = optuna.create_study(
    direction="maximize"
)

study_cat.optimize(
    objective_cat,
    n_trials=50,
    show_progress_bar=True
)

print(
    f"\n✅ Best CatBoost PR-AUC: "
    f"{study_cat.best_value:.4f}"
)

print(
    f"Best params:\n{study_cat.best_params}"
)

  0%|          | 0/50 [00:00<?, ?it/s]


✅ Best CatBoost PR-AUC: 0.3628
Best params:
{'iterations': 456, 'depth': 4, 'learning_rate': 0.29571944621864876, 'l2_leaf_reg': 4.65523530138566, 'random_strength': 1.2710203380836955}


In [11]:
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# =========================
# Tuned XGBoost
# =========================

xgb_best = XGBClassifier(
    n_estimators=291,
    max_depth=10,
    learning_rate=0.012168849729434708,
    subsample=0.6199915172596526,
    colsample_bytree=0.9652638607771129,
    min_child_weight=2,
    eval_metric="aucpr",
    random_state=42,
    verbosity=0
)

# =========================
# Tuned LightGBM
# =========================

lgbm_best = LGBMClassifier(
    n_estimators=367,
    max_depth=4,
    learning_rate=0.25614513269902417,
    subsample=0.9678428280283313,
    colsample_bytree=0.5746492798628466,
    min_child_samples=56,
    num_leaves=107,
    random_state=42,
    verbosity=-1
)

# =========================
# Tuned CatBoost
# =========================

cat_best = CatBoostClassifier(
    iterations=456,
    depth=4,
    learning_rate=0.29571944621864876,
    l2_leaf_reg=4.65523530138566,
    random_strength=1.2710203380836955,
    random_state=42,
    verbose=0
)

print("✅ Tuned models created")

✅ Tuned models created


In [12]:
xgb_best.fit(X_train_sm, y_train_sm)

lgbm_best.fit(X_train_sm, y_train_sm)

cat_best.fit(X_train_sm, y_train_sm)

print("✅ All tuned models trained")

✅ All tuned models trained


In [13]:
results_tuned = []

results_tuned.append(
    evaluate_model(
        "Tuned XGBoost",
        xgb_best,
        X_test,
        y_test
    )
)

results_tuned.append(
    evaluate_model(
        "Tuned LightGBM",
        lgbm_best,
        X_test,
        y_test
    )
)

results_tuned.append(
    evaluate_model(
        "Tuned CatBoost",
        cat_best,
        X_test,
        y_test
    )
)

results_tuned_df = pd.DataFrame(results_tuned)

results_tuned_df.sort_values(
    by="PR-AUC",
    ascending=False
)


  Tuned XGBoost
  ROC-AUC : 0.8540
  PR-AUC  : 0.3451  ← primary metric
  F1      : 0.4140

              precision    recall  f1-score   support

           0       0.96      0.95      0.96     27995
           1       0.40      0.43      0.41      2005

    accuracy                           0.92     30000
   macro avg       0.68      0.69      0.68     30000
weighted avg       0.92      0.92      0.92     30000


  Tuned LightGBM
  ROC-AUC : 0.8542
  PR-AUC  : 0.3571  ← primary metric
  F1      : 0.3270

              precision    recall  f1-score   support

           0       0.95      0.98      0.96     27995
           1       0.50      0.24      0.33      2005

    accuracy                           0.93     30000
   macro avg       0.72      0.61      0.65     30000
weighted avg       0.92      0.93      0.92     30000


  Tuned CatBoost
  ROC-AUC : 0.8565
  PR-AUC  : 0.3628  ← primary metric
  F1      : 0.3393

              precision    recall  f1-score   support

          

,Model,ROC-AUC,PR-AUC,F1
2,Tuned CatBoost,0.856521,0.362813,0.339274
1,Tuned LightGBM,0.854192,0.357076,0.326968
0,Tuned XGBoost,0.854044,0.345070,0.414023


In [14]:
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

In [15]:
n_train = len(X_train)

oof_xgb = np.zeros(n_train)
oof_lgbm = np.zeros(n_train)
oof_cat = np.zeros(n_train)

test_xgb = np.zeros(len(X_test))
test_lgbm = np.zeros(len(X_test))
test_cat = np.zeros(len(X_test))

print("✅ OOF arrays initialized")

✅ OOF arrays initialized


In [16]:
kf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("✅ 5-Fold CV ready")

✅ 5-Fold CV ready


In [17]:
from imblearn.over_sampling import SMOTE

for fold, (train_idx, valid_idx) in enumerate(
    kf.split(X_train, y_train),
    start=1
):

    print(f"\n========== Fold {fold}/5 ==========")

    # -------------------------
    # Split fold
    # -------------------------

    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]

    X_val = X_train.iloc[valid_idx]
    y_val = y_train.iloc[valid_idx]

    # -------------------------
    # SMOTE ONLY ON TRAIN FOLD
    # -------------------------

    smote = SMOTE(
        random_state=42,
        k_neighbors=5
    )

    X_tr_sm, y_tr_sm = smote.fit_resample(
        X_tr,
        y_tr
    )

    # -------------------------
    # XGBoost
    # -------------------------

    xgb_fold = XGBClassifier(
        **study_xgb.best_params,
        eval_metric="aucpr",
        random_state=42,
        verbosity=0
    )

    xgb_fold.fit(X_tr_sm, y_tr_sm)

    oof_xgb[valid_idx] = (
        xgb_fold.predict_proba(X_val)[:, 1]
    )

    test_xgb += (
        xgb_fold.predict_proba(X_test)[:, 1] / 5
    )

    # -------------------------
    # LightGBM
    # -------------------------

    lgbm_fold = LGBMClassifier(
        **study_lgbm.best_params,
        random_state=42,
        verbosity=-1
    )

    lgbm_fold.fit(X_tr_sm, y_tr_sm)

    oof_lgbm[valid_idx] = (
        lgbm_fold.predict_proba(X_val)[:, 1]
    )

    test_lgbm += (
        lgbm_fold.predict_proba(X_test)[:, 1] / 5
    )

    # -------------------------
    # CatBoost
    # -------------------------

    cat_fold = CatBoostClassifier(
        **study_cat.best_params,
        random_state=42,
        verbose=0
    )

    cat_fold.fit(X_tr_sm, y_tr_sm)

    oof_cat[valid_idx] = (
        cat_fold.predict_proba(X_val)[:, 1]
    )

    test_cat += (
        cat_fold.predict_proba(X_test)[:, 1] / 5
    )

print("\n✅ OOF predictions complete")


========== Fold 1/5 ==========

========== Fold 2/5 ==========

========== Fold 3/5 ==========

========== Fold 4/5 ==========

========== Fold 5/5 ==========

✅ OOF predictions complete


In [18]:
# =========================
# Training (OOF predictions)
# =========================

X_stack_train = pd.DataFrame({
    "xgb_pred": oof_xgb,
    "lgbm_pred": oof_lgbm,
    "cat_pred": oof_cat
})

# =========================
# Test predictions (averaged)
# =========================

X_stack_test = pd.DataFrame({
    "xgb_pred": test_xgb,
    "lgbm_pred": test_lgbm,
    "cat_pred": test_cat
})

print("X_stack_train shape:", X_stack_train.shape)
print("X_stack_test shape:", X_stack_test.shape)

X_stack_train shape: (120000, 3)
X_stack_test shape: (30000, 3)


In [19]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

meta_model.fit(X_stack_train, y_train)

print("✅ Meta-learner trained")

✅ Meta-learner trained


In [20]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

final_proba = meta_model.predict_proba(X_stack_test)[:, 1]
final_pred = (final_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, final_proba)
pr_auc = average_precision_score(y_test, final_proba)
f1 = f1_score(y_test, final_pred)

print("\n==============================")
print("   STACKING ENSEMBLE RESULT")
print("==============================")
print(f"ROC-AUC : {roc_auc:.4f}")
print(f"PR-AUC  : {pr_auc:.4f}")
print(f"F1      : {f1:.4f}")


   STACKING ENSEMBLE RESULT
ROC-AUC : 0.8588
PR-AUC  : 0.3558
F1      : 0.3698


In [21]:
import joblib
import json
import numpy as np
import os

# ── Create directories if they don't exist ──────────────────────────
os.makedirs(f"{MODELS_PATH}", exist_ok=True)
os.makedirs(f"{DATA_PROCESSED}", exist_ok=True)
os.makedirs(f"{PROJECT_ROOT}/configs", exist_ok=True)

# ── 1 & 2. Save all models ──────────────────────────────────────────
joblib.dump(xgb_best,    f"{MODELS_PATH}/xgb_best.pkl")
joblib.dump(lgbm_best,   f"{MODELS_PATH}/lgbm_best.pkl")
joblib.dump(cat_best,    f"{MODELS_PATH}/cat_best.pkl")
joblib.dump(meta_model,  f"{MODELS_PATH}/meta_model.pkl")

print("✅ Models saved")

# ── 3. Save stacked predictions ─────────────────────────────────────
X_stack_train.to_csv(f"{DATA_PROCESSED}/X_stack_train.csv", index=False)
X_stack_test.to_csv(f"{DATA_PROCESSED}/X_stack_test.csv",  index=False)

print("✅ Stacked predictions saved")

# ── 4. Save OOF arrays ───────────────────────────────────────────────
np.savez(
    f"{DATA_PROCESSED}/oof_arrays.npz",
    oof_xgb=oof_xgb,
    oof_lgbm=oof_lgbm,
    oof_cat=oof_cat
)

print("✅ OOF arrays saved")

# ── 5. Save best Optuna params ───────────────────────────────────────
best_params = {
    "xgb":  study_xgb.best_params,
    "lgbm": study_lgbm.best_params,
    "cat":  study_cat.best_params
}

with open(f"{PROJECT_ROOT}/configs/best_params.json", "w") as f:
    json.dump(best_params, f, indent=2)

print("✅ Best params saved")
print("\n🎯 All Phase 6 artifacts saved successfully!")

✅ Models saved
✅ Stacked predictions saved
✅ OOF arrays saved
✅ Best params saved

🎯 All Phase 6 artifacts saved successfully!
